# Session 5 — RAG with LangChain, Gemini Embeddings & Qdrant

Northstar can now look up accounts and invoices, but it can still paraphrase company policy incorrectly. Today we build a searchable knowledge base, retrieve the right passages, answer with sources, and finally give retrieval to an agent as a tool.

By the end you can explain and build the full pipeline: **load → split → embed → store → retrieve → generate**.

## 1. Setup

🎯 **Purpose:** install the integrations used in the live examples. Docling is intentionally separate and heavy; install it only when you want to run that section.

> Restart the notebook kernel after the first install if an import still uses an older package version.

In [ ]:
%pip install -q -U langchain langgraph langchain-google-genai langchain-community \
    langchain-text-splitters langchain-qdrant qdrant-client python-dotenv google-genai

# Optional document loaders (Docling downloads/loads ML models and can take time):
%pip install -q -U langchain-pymupdf4llm
# %pip install -q -U langchain-docling

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import os

for name in ["GEMINI_API_KEY", "QDRANT_URL", "QDRANT_API_KEY"]:
    print(f"{name}: {'set' if os.getenv(name) else 'MISSING'}")
# Never print an API key itself.

## 2. What retrieval and RAG actually do

LLMs have finite context and static training knowledge. **Retrieval** fetches relevant external knowledge at query time. **RAG** (Retrieval-Augmented Generation) places that evidence in front of a model so it can produce a grounded answer.

The building blocks are modular:

| Block | Job | Output |
|---|---|---|
| Document loader | Read a source in a standard form | `Document(page_content, metadata)` |
| Text splitter | Make independently retrievable chunks | smaller `Document`s |
| Embedding model | Map meaning to numbers | a fixed-length vector |
| Vector store | Persist vectors and search nearest neighbors | relevant chunks |
| Retriever | Standard query → documents interface | `list[Document]` |
| Generator | Answer only from retrieved context | grounded response + sources |

Three common architectures:

- **2-step RAG:** always retrieve, then generate. Predictable and fast.
- **Agentic RAG:** an agent decides whether and which source to retrieve. Flexible, but latency and behavior vary.
- **Hybrid RAG:** adds query rewriting, retrieval checks, or answer validation.

Reference: [LangChain retrieval overview](https://docs.langchain.com/oss/python/langchain/retrieval).

## 3. Document loaders: one `Document` interface

Loaders hide source-specific parsing and return the same LangChain `Document` shape. Most expose:

- `load()` — parse everything and return a list.
- `lazy_load()` — yield documents as they become available, which reduces peak memory for large collections.

Choose **one** for a run. Calling both parses the source twice. See the [loader integration catalog](https://docs.langchain.com/oss/python/integrations/document_loaders).

### 3.1 Docling — structure-aware parsing

Docling parses PDFs, DOCX, PPTX, HTML, and more into a rich representation that understands layout and tables. Its LangChain loader defaults to document chunks and can alternatively export one Markdown document per input.

Why can it take a long time?

1. The first run may download and initialize PyTorch-backed models.
2. It performs layout analysis and table understanding, not only plain-text extraction.
3. Scanned pages can require OCR; more pages and high-resolution images add work.
4. CPU inference is slower. The integration docs recommend a GPU-enabled runtime for best conversion speed.
5. A remote PDF must be downloaded before it is parsed.

Use Docling when document structure matters. Cache converted output instead of reparsing unchanged files on every app start. References: [Docling installation](https://docling-project.github.io/docling/getting_started/installation/) and [LangChain Docling loader](https://docs.langchain.com/oss/python/integrations/document_loaders/docling).

In [ ]:
from langchain_docling.loader import DoclingLoader

# This parses a real research paper and can be slow on its first run.
RUN_DOCLING = False

if RUN_DOCLING:
    file_path = "https://arxiv.org/pdf/2408.09869"
    loader = DoclingLoader(file_path=file_path)

    # Choose ONE:
    documents = loader.load()
    # documents = list(loader.lazy_load())  # lower peak memory for many/large inputs

    print(f"Loaded {len(documents)} document chunk(s)")
    print(documents[0].page_content[:500])

### 3.2 PyMuPDF4LLM — a fast PDF-first option

PyMuPDF4LLM is a practical choice for PDFs when you want fast Markdown-oriented extraction. It supports page/single-document modes and optional image/table extraction. It needs no API credential.

The original live snippet imported `t`; the correct class is `PyMuPDF4LLMLoader`. Avoid hard-coded absolute instructor paths so every student can run the cell. Reference: [PyMuPDF4LLMLoader integration](https://docs.langchain.com/oss/python/integrations/document_loaders/pymupdf4llm).

In [ ]:
from pathlib import Path
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

pdf_path = Path("data/test.pdf")  # put your PDF here
if pdf_path.exists():
    loader = PyMuPDF4LLMLoader(str(pdf_path))
    pdf_docs = loader.load()
    print(f"Loaded {len(pdf_docs)} page document(s)")
    display(pdf_docs[0])
else:
    print(f"Add a PDF at {pdf_path.resolve()} to run this example.")

### Loader choice is an engineering trade-off

| Need | Good starting point |
|---|---|
| Fast extraction from clean, digital PDFs | PyMuPDF4LLM |
| Complex layout, tables, mixed office formats | Docling |
| Many files with bounded memory | a loader's `lazy_load()` |
| Scans | a loader/pipeline with OCR enabled |

Always inspect the extracted text and metadata. A sophisticated retriever cannot recover content the loader lost.

## 4. Text splitters — create useful retrieval units

A whole book is too broad to retrieve; tiny fragments lose meaning. `RecursiveCharacterTextSplitter` is LangChain's recommended starting point because it tries paragraphs, then sentences/words, keeping larger natural units intact when possible.

- `chunk_size`: maximum approximate characters per chunk for this splitter.
- `chunk_overlap`: repeated boundary context that reduces cut-off facts, but increases storage and duplicate retrieval.
- `add_start_index=True`: records where each chunk started, useful for debugging/citations.

Reference: [LangChain text splitters](https://docs.langchain.com/oss/python/integrations/splitters).

In [ ]:
text = """Text is naturally organized into hierarchical units such as paragraphs, sentences, and words.
We can leverage this structure to maintain natural language flow and semantic coherence.

RecursiveCharacterTextSplitter attempts to keep larger units such as paragraphs intact.
If a unit exceeds the chunk size, it moves to smaller units such as sentences, then words."""

from langchain_text_splitters import RecursiveCharacterTextSplitter

small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
texts = small_splitter.split_text(text)
for i, chunk in enumerate(texts, start=1):
    print(f"CHUNK {i} ({len(chunk)} chars): {chunk!r}\n")

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for loyalty and friendliness.",
        metadata={"source": "mammal-pets.md", "section": "Dogs"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets.md", "section": "Cats"},
    ),
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1_000, chunk_overlap=200, add_start_index=True
)
all_splits = splitter.split_documents(documents)
print(f"{len(documents)} documents -> {len(all_splits)} chunks")
display(all_splits[0])

## 5. Embeddings — meaning as coordinates

An embedding model maps content to a fixed-length numeric vector. Similar meanings should have nearby vectors, letting semantic search find *refund timing* even when a passage says *bank posting time*.

Use the **same model and dimensionality** for indexed documents and later queries. Changing the embedding model creates an incompatible vector space and requires re-indexing.

Reference: [LangChain embedding integrations](https://docs.langchain.com/oss/python/integrations/embeddings).

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")
vector_1 = embeddings.embed_query(documents[0].page_content)
vector_2 = embeddings.embed_query(documents[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}")
print(vector_1[:10])

In [ ]:
# Cosine similarity: direction, not raw magnitude. Closer to 1 means more aligned.
from math import sqrt

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sqrt(sum(x * x for x in a))
    norm_b = sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)

print(f"dog ↔ cat similarity: {cosine_similarity(vector_1, vector_2):.3f}")

### Gemini Embedding 2: the latest multimodal embedding model

`gemini-embedding-2` maps text, images, video, audio, and PDFs into one embedding space. It supports 128–3072 output dimensions (Google recommends 768, 1536, or 3072) and an 8,192-token overall input limit. PDFs are limited to one file and six pages per request.

Important boundary: LangChain's standard `Embeddings` interface is currently text-focused. Use `GoogleGenerativeAIEmbeddings` for the text RAG pipeline; use Google's native Gen AI SDK when embedding an image, audio/video, or a PDF directly.

For text retrieval with Embedding 2, Google recommends instructions in the input: query as `task: search result | query: ...`, and documents as `title: ... | text: ...`. Unlike Embedding 1, Embedding 2 does not accept `task_type`. See [Google's Embeddings guide](https://ai.google.dev/gemini-api/docs/embeddings).

In [ ]:
# Optional native multimodal example: text + image -> one aggregated embedding.
from pathlib import Path
from google import genai
from google.genai import types

image_path = Path("data/sample.png")
if image_path.exists():
    client_genai = genai.Client()
    result = client_genai.models.embed_content(
        model="gemini-embedding-2",
        contents=[
            types.Part.from_text(text="Northstar product screenshot"),
            types.Part.from_bytes(data=image_path.read_bytes(), mime_type="image/png"),
        ],
        config=types.EmbedContentConfig(output_dimensionality=768),
    )
    print(len(result.embeddings[0].values))
else:
    print("Add data/sample.png to run the multimodal example.")

## 6. Qdrant Cloud — store and search vectors

A vector store persists embeddings plus each chunk's text/metadata and performs nearest-neighbor search. LangChain gives a common interface: `add_documents`, `delete`, and `similarity_search`.

Qdrant Cloud setup:

1. Create a free cluster in [Qdrant Cloud](https://cloud.qdrant.io/).
2. Copy the cluster URL and create an API key.
3. Put them in `.env` as `QDRANT_URL` and `QDRANT_API_KEY`—never paste secrets into a notebook.
4. A collection fixes its vector size and distance metric. A different embedding dimension needs a new collection or a deliberate rebuild.

References: [vector stores](https://docs.langchain.com/oss/python/integrations/vectorstores) and [Qdrant integration](https://docs.langchain.com/oss/python/integrations/vectorstores/qdrant).

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
if not qdrant_url or not qdrant_api_key:
    raise RuntimeError("Set QDRANT_URL and QDRANT_API_KEY in .env first.")

client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key)
collection_name = "session5_demo_v1"
vector_size = len(embeddings.embed_query("dimension probe"))

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )

vector_store = QdrantVectorStore(
    client=client, collection_name=collection_name, embedding=embeddings
)
print(f"Ready: {collection_name} ({vector_size} dimensions)")

### Index documents with stable IDs

Random IDs duplicate data every time a teaching cell is rerun. Deterministic IDs overwrite the same points, making this small ingestion example idempotent. Production pipelines should also record document versions and delete chunks that disappeared from a newer version.

In [ ]:
from uuid import NAMESPACE_URL, uuid5

ids = [
    str(uuid5(NAMESPACE_URL, f"{doc.metadata}|{doc.page_content}"))
    for doc in all_splits
]
vector_store.add_documents(documents=all_splits, ids=ids)
print(f"Indexed {len(ids)} chunk(s)")

### Search directly and inspect the evidence

Retrieval quality must be checked *before* generation. If the right passage is absent, a polished model answer is still ungrounded. Qdrant cosine similarity scores are larger for closer matches, but score meaning depends on store/metric—do not copy one universal threshold across systems.

In [ ]:
results = vector_store.similarity_search_with_score(
    "Which pet is loyal and friendly?", k=2
)
for doc, score in results:
    print(f"score={score:.3f} source={doc.metadata.get('source')}")
    print(doc.page_content, "\n")

In [ ]:
# The retriever is the standard query -> Documents adapter used by chains and workflows.
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("Tell me about independent pets")
for doc in retrieved_docs:
    print(doc.page_content)

## 7. Two-step RAG — retrieve, then always generate

The safest minimal pattern has a visible boundary: retrieval supplies passages; the prompt tells the model those passages are the only evidence. Include source metadata in the context rather than asking the model to invent citations.

In [ ]:
from langchain.chat_models import init_chat_model

MODEL = "google_genai:gemini-3.1-flash-lite"
model = init_chat_model(MODEL)

def answer_with_rag(question: str) -> str:
    found = retriever.invoke(question)
    context = "\n\n".join(
        f"SOURCE: [{d.metadata.get('source', 'unknown')}]\n{d.page_content}"
        for d in found
    )
    prompt = f"""Answer only from CONTEXT. If it is insufficient, say so.
Cite factual claims using the supplied [source].

QUESTION:
{question}

CONTEXT:
{context}"""
    return model.invoke(prompt).text

print(answer_with_rag("Which pet is described as independent?"))

## 8. Agentic RAG — retrieval becomes a tool

In 2-step RAG, retrieval always runs. In agentic RAG, the agent reads tool names/descriptions and decides whether retrieval is needed. Separate tools are valuable when sources have different authority or topics—for example, Northstar's refund policy and company policy.

A retrieval tool should return **content plus source metadata**. Returning only prose makes grounded citations impossible.

### End-to-end with the Avengers PDF

The live snippet used an instructor-specific absolute path. Put your own copy at `data/Avengers_The_Light_Beyond_the_Storm.pdf`; this cell loads it, chunks it, and indexes a separate collection. Keeping the book separate from the pet demo avoids mixing unrelated knowledge bases.

In [ ]:
book_path = Path("data/Avengers_The_Light_Beyond_the_Storm.pdf")
book_vector_store = None

if book_path.exists():
    book_docs = PyMuPDF4LLMLoader(str(book_path)).load()
    book_splits = RecursiveCharacterTextSplitter(
        chunk_size=1_000, chunk_overlap=200, add_start_index=True
    ).split_documents(book_docs)

    book_collection = "session5_avengers_v1"
    if not client.collection_exists(book_collection):
        client.create_collection(
            collection_name=book_collection,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
    book_vector_store = QdrantVectorStore(
        client=client, collection_name=book_collection, embedding=embeddings
    )
    book_ids = [
        str(uuid5(NAMESPACE_URL, f"{d.metadata}|{d.page_content}"))
        for d in book_splits
    ]
    book_vector_store.add_documents(book_splits, ids=book_ids)
    print(f"Indexed {len(book_splits)} book chunks")
else:
    print(f"Add the book PDF at {book_path.resolve()} before running the agent demo.")

In [ ]:
from langchain.tools import tool

@tool
def query_avengers_knowledge(query: str) -> str:
    """Search the indexed Avengers book for chapters, characters, events, or dialogue."""
    if book_vector_store is None:
        return "The Avengers PDF has not been indexed. Add the file and run the ingestion cell."
    hits = book_vector_store.similarity_search_with_score(query, k=4)
    if not hits:
        return "No relevant passage found."

    output = []
    for rank, (doc, score) in enumerate(hits, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")
        output.append(
            f"SOURCE {rank}: [{source}, page {page}]\n"
            f"SIMILARITY: {score:.3f}\n{doc.page_content}"
        )
    return "\n\n".join(output)

print(query_avengers_knowledge.invoke({"query": "Give me the name of chapter two"}))

In [ ]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model=MODEL,
    tools=[query_avengers_knowledge],
    system_prompt=(
        "Use query_avengers_knowledge for questions about the indexed book. "
        "Answer only from retrieved passages and cite the returned source label. "
        "If no passage supports an answer, say you could not find it."
    ),
)
result = rag_agent.invoke({"messages": [{"role": "user", "content": (
    "Complete Bruce's dialogue in the Epilogue that begins 'The machine thought strength ...'"
)}]})
print(result["messages"][-1].text)

## 9. What can fail? Debug retrieval before blaming the model

| Symptom | Likely layer | First check |
|---|---|---|
| Answer misses a fact | loader | Is the fact present in extracted text? |
| Relevant paragraph is split apart | splitter | Inspect chunk boundaries and overlap |
| Wrong passages rank highly | embedding/query | Try real questions; inspect top-k and scores |
| Old/duplicate facts appear | ingestion | Use stable IDs, versions, and deletion strategy |
| Agent uses wrong collection | tool design | Improve tool name, docstring, and system routing rules |
| Citation looks real but is false | generation | Supply metadata; require citations only from tool output |
| Cloud query fails | infrastructure | Check URL/key, collection dimension, network, and Qdrant logs |

Trace the loader inputs, chunk contents, retrieval query, returned passages/scores, tool call, and final answer in LangSmith. A RAG trace should make the evidence path inspectable.

## 11. Recap

- RAG is an evidence pipeline, not merely a vector database.
- Loader quality and chunk boundaries determine what retrieval can possibly find.
- Gemini Embedding 2 supports multimodal input natively; LangChain's standard embedding interface remains text-focused.
- Qdrant stores chunks, embeddings, and metadata and returns semantic neighbors.
- Two-step RAG always retrieves; agentic RAG exposes retrieval as one or more tools.
- Grounded citations come from retrieved metadata, and retrieval must be evaluated separately from generation.